In [7]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH10-Retriever")

loader = WebBaseLoader(
    "https://teddylee777.github.io/openai/openai-assistant-tutorial/", encoding="utf-8"
)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = loader.load_and_split(text_splitter)

openai_embedding = OpenAIEmbeddings()

db = FAISS.from_documents(docs, openai_embedding)

retriever = db.as_retriever()

query = "OpenAI Assistant API의 Functions 사용법에 대해 알려주세요."
relevant_docs = retriever.invoke(query)

len(relevant_docs)
print(relevant_docs[1].page_content)

LangSmith 추적을 시작합니다.
[프로젝트명]
CH10-Retriever
가장 강력한 도구로서, Assistant에게 사용자 정의 함수를 지정할 수 있습니다. 이는 Chat Completions API에서의 함수 호출과 매우 유사합니다.


Function calling(함수 호출) 도구를 사용하면 Assistant 에게 사용자 정의 함수 를 설명하여 호출해야 하는 함수를 인자와 함께 지능적으로 반환하도록 할 수 있습니다.


Assistant API는 실행 중에 함수를 호출할 때 실행을 일시 중지하며, 함수 호출 결과를 다시 제공하여 Run 실행을 계속할 수 있습니다. (이는 사용자 피드백을 받아 재게할 수 있는 의미이기도 합니다. 아래 튜토리얼에서 상세히 다룹니다).


In [8]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0, model='gpt-4o-mini')

#MultiQueryRetriever 
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(),
    llm=llm,
)


In [9]:
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [10]:
question = "OpenAI Assistant API의 Functions 사용법에 대해 알려주세요."

relevant_docs = multiquery_retriever.invoke(question)

print(
    f"============\n검색된 문서 개수: {len(relevant_docs)}",
    end="\n============\n",
)

print(relevant_docs[0].page_content)

검색된 문서 개수: 4
OpenAI의 새로운 Assistants API는 대화와 더불어 강력한 도구 접근성을 제공합니다. 본 튜토리얼은 OpenAI Assistants API를 활용하는 내용을 다룹니다. 특히, Assistant API 가 제공하는 도구인 Code Interpreter, Retrieval, Functions 를 활용하는 방법에 대해 다룹니다. 이와 더불어 파일을 업로드 하는 내용과 사용자의 피드백을 제출하는 내용도 튜토리얼 말미에 포함하고 있습니다.



주요내용


In [11]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

#프롬프트 템플릿을 정의(다섯 개의 질문을 생성하도록 프롬프트를 작성)
prompt = PromptTemplate.from_template(
    """You are an AI language model assistant.
    Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database.
    By generating multiple perspectives on the user question, your goal is to help the user overcome some of the limitations of the distance-based similarity search.
    Your response should be a listof values separated by ne lines, eg: `foo\nbar\nbaz\n`
    
    #ORIGINAL QUESTION:
    {question}
    
    #Answer in Korean:
    """
)

In [14]:
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

custom_multiquery_chain = (
    {"question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
)

question = "OpenAI Assistant API의 Functions 사용법에 대해 알려주세요."

multi_queries = custom_multiquery_chain.invoke(question)
multi_queries

'OpenAI Assistant API의 Functions 기능을 사용하는 방법은 무엇인가요?\nOpenAI Assistant API에서 Functions를 활용하는 절차를 설명해 주세요.\nOpenAI Assistant API의 Functions 사용에 대한 가이드를 제공해 주실 수 있나요?\nOpenAI Assistant API의 Functions 기능을 어떻게 적용할 수 있는지 알고 싶습니다.\nOpenAI Assistant API에서 Functions를 사용하는 예시를 보여주세요.'

In [15]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    llm=custom_multiquery_chain,
    retriever=db.as_retriever(),
)

In [16]:
relevant_docs = multiquery_retriever.invoke(question)

print(
    f"==============\n검색된 문서 개수: {len(relevant_docs)}",
    end="\n==============\n",
)
print(relevant_docs[0].page_content)

검색된 문서 개수: 5
OpenAI의 새로운 Assistants API는 대화와 더불어 강력한 도구 접근성을 제공합니다. 본 튜토리얼은 OpenAI Assistants API를 활용하는 내용을 다룹니다. 특히, Assistant API 가 제공하는 도구인 Code Interpreter, Retrieval, Functions 를 활용하는 방법에 대해 다룹니다. 이와 더불어 파일을 업로드 하는 내용과 사용자의 피드백을 제출하는 내용도 튜토리얼 말미에 포함하고 있습니다.



주요내용
